# Process the sqY2H data

This is the main PPI profiling assay. This script loads the raw scores, normalizes them, and produces edgotyping profiles. Throughout the script, "ad" means "activation domain" and "db" means "DNA binding domain" of a transcription factor. A protein is fused to each of these; if the two proteins interact, then the TF is able to up-regulate the expression of a gene that allows the yeast to survive on the media. Thus, the yeast growth score is correlated to the protein-protein interaction strength.

In [ ]:
import pandas as pd 
import polars as pl
from striprtf.striprtf import rtf_to_text
import re

In [2]:
inputs_dir = "../1_inputs/sqY2H"
outputs_dir = "../3_outputs"

In [46]:
# read in a list of ORF IDs and swim seq validation results
raw_scores = pd.read_table(f"{inputs_dir}/VarChAMP_1percent_Y2H_scores.tsv")
raw_scores = raw_scores[[
    "growth_score_id", "scoring_pla", "scoring_pos", "manual_score_growth"
]]
raw_scores[['retest_batch', 'condition', 'retest_pla']] = raw_scores['scoring_pla'].str.extract(r"([A-Z0-9]+)(r\d+)_0*(\d+)")
raw_scores['retest_pla'] = raw_scores['retest_pla'].astype(int)
raw_scores.shape

(248192, 7)

In [ ]:
# Make the scores match an allele to the 96 wells of the source plate 
allele_map = pd.read_csv(f"{inputs_dir}/cp_files/mapping_y2h.csv").drop(columns=['ad_symbol', 'db_symbol'])

# create dictionary mapping 96 well position to 384 well position
with open(f"{inputs_dir}/mapping_384_to_96.rtf", "r") as f:
    rtf_content = f.read()

plain_text = rtf_to_text(rtf_content)
match = re.search(r"mapping_384_to_96_d\s*=\s*({.*})", plain_text, re.DOTALL)
dict_str = match.group(1)
dict_str = dict_str.replace("\n", "").replace("\r", "")
map_384_96 = eval(dict_str)
map_df = pd.DataFrame(list(map_384_96.items()), columns=['scoring_pos', 'retest_pos'])

In [ ]:

# Map the 96-pos to 384-pos
allele_map = allele_map.merge(map_df, on="retest_pos")
allele_map['retest_pla'] = allele_map['retest_pla'].astype(int)

### what does db_mut_id equal 99999 mean? -> RRS scores that should be checked for QA/QC

In [ ]:
# add orf ids to scores
scores = raw_scores.merge(allele_map, on=["scoring_pos", "retest_batch", "retest_pla"], how="left")
print(scores.shape)

# Filter out r01-r06 and convert to other condition names
exclude_list = ["r01", "r02", "r03", "r04", "r05", "r06"]
scores = scores[~scores['condition'].isin(exclude_list)]
print(scores.shape)

# get only plates from relevant batches
scores = scores[scores.standard_batch.isin(["VUSAPWT1B1","VUSAPWT1B2","VUSAPWT2B1","VUSAPWT6B1"])]
print(scores.shape)

# both db_orf_id and ad_orf_id must have values (otherwise they are empty wells)
scores = scores[(~scores.db_orf_id.isnull())&(~scores.ad_orf_id.isnull())]
print(scores.shape)

#### write out raw individual scores with matched orf ids

(248192, 19)
(243584, 19)
(231969, 19)
(229475, 19)


In [ ]:
# ad orf, ad mut, db orf, db mut, condition
# calc num obs and most freq score
scores = pl.DataFrame(scores)
consensus_scores = scores.group_by(['ad_orf_id', 'ad_mut_id', 'db_orf_id', 'db_mut_id', 'condition', 'retest_batch']).agg(
    pl.col("manual_score_growth").mode().alias("consensus_score_list"),
    pl.len().alias("row_count")
)

consensus_scores = consensus_scores.with_columns(
    pl.col("consensus_score_list").list.len().alias("num_most_freq")
)

consensus_easy = consensus_scores.filter(pl.col("num_most_freq") != 2)
consensus_easy = consensus_easy.with_columns(
    pl.when(pl.col("num_most_freq") == 1)
      .then(pl.col("consensus_score_list").list.first())
      .when(pl.col("num_most_freq") > 2)
      .then(pl.lit(99999))
      .otherwise(None)
      .alias("consensus_score")
)

consensus_hard = consensus_scores.filter(pl.col("num_most_freq") == 2)
consensus_hard = consensus_hard.with_columns(
    (pl.col("consensus_score_list").list.get(0) - pl.col("consensus_score_list").list.get(1))
    .abs()
    .alias("score_diff")
)
consensus_hard = consensus_hard.with_columns(
    pl.when(pl.col("score_diff") > 1)
      .then(pl.lit(99999))
      .when(pl.col("score_diff").is_null())
      .then(
          pl.col("consensus_score_list")
            .list.eval(pl.element().drop_nulls())
            .list.first()
      )
      .when(pl.col("score_diff") == 1)
      .then(
          pl.col("consensus_score_list")
            .list.eval(pl.element().max())
            .list.first()
      )
      .otherwise(None)
      .alias("consensus_score")
).drop("score_diff")

consensus_scores = pl.concat([consensus_easy, consensus_hard]).drop([
    "consensus_score_list", "row_count", "num_most_freq"
])

#### write out the raw consensus scores

In [135]:
consensus_scores

ad_orf_id,ad_mut_id,db_orf_id,db_mut_id,condition,retest_batch,consensus_score
f64,f64,f64,f64,str,str,f64
54914.0,null,54914.0,28866.0,"""r11""","""VUSAPWT1B1""",0.0
152.0,null,10122.0,0.0,"""r12""","""VUSAPWT1B2""",3.0
6534.0,null,8420.0,216661.0,"""r11""","""VUSAPWT1B2""",0.0
13795.0,null,8416.0,216633.0,"""r08""","""VUSAPWT1B2""",0.0
1.00002625e8,null,53030.0,218690.0,"""r08""","""VUSAPWT1B2""",null
…,…,…,…,…,…,…
4105.0,null,12573.0,2229.0,"""r08""","""VUSAPWT1B1""",4.0
0.0,null,4723.0,201725.0,"""r09""","""VUSAPWT1B1""",2.0
70264.0,null,2713.0,953.0,"""r09""","""VUSAPWT1B1""",2.0


In [ ]:
# more than 2 or 2 that are not consecutive: failure = 9999999
# if 2 and nan, take the value
# for the remaining with 2 consecutive, take the max

In [ ]:
condition_dict = {
    "r07": "LW",
    "r08": "LWA",
    "r09": "LWH1",
    "r10": "LWH10",
    "r11": "LWH25",
    "r12": "LWAH1",
    "r13": "LW",
    "r14": "LWA",
    "r15": "LWH1",
    "r16": "LWH10",
    "r17": "LWH25",
    "r18": "LWAH1"
}

In [ ]:
# write out raw score

In [ ]:
# each interaction tested 4 times (robotic technical replicates)
# want to get one score
# Flo wants most frequent score, not the average (this could be the mean; we can adjust later if needed)
# only need manual_score

# Each allele has 4 spots, this turns to one score

# Extract conditions and put in a column (r01-r18), add scorer, and the condition type (1-6 and name)
# Filter out r01-r06

# Write out this file

In [ ]:
### Normalization - filters & final score

# 1 - Cells should either be all there or all not. Control media - are the two proteins present
# 3 - 5, 2: strength of the interaction from low to high. 2 condition has a different media where it's harder for yeast to grow.
# 6 - combination of 1 & 2 medias, but very complicated to logically process so currently not used but retained in case we use it in the future

# 16 possible highest score: 4/4 for each of 3, 4, 5, 2, then divide by 16

# dbX, empty abY. If yeast can still grow, it's called autoactivation
# DB-X+0AD, DB-X+AD-Y. We get score for both. If both equal 4, then we discard. If 0 and 4, worked well. 
# If somewhere in between (ie. 2 and 4), we sometimes filter out, sometimes keep with a score adjustment (check Georges code)
# Indicated with ad_orf_id (probably equal to 0 for empty; verify in Georges code)

# Postitions with both empty ad and db orf ids are media controls. 
# 6*96 well plate = 6*4 positions on 384, 6*384 on mega plate, but ctrls only spotted once per mega plate

# Instructions:
# check media controls for each plate: 24 spots / mega plate, must have score of X to keep that mega plate. Each control of the six must be checked separately - talk to Flo but implement after.
# drop 6
# filter by 1 - if no score, then remove that position from the dataset
# filter by empty AD
# select conditions of interest (ie. 3-5 or 3-5,2), then combine scores and normalize by 4*(# selected conditions)
# read in sequence confirmation results and discard everything that's not sequence confirmed
# write out normalized scores

In [ ]:
### Edgotyping profile

# write out profile